In [1]:
import numpy as np
from scipy.special import jv, jn_zeros
from scipy.sparse.linalg import eigsh, ArpackNoConvergence
from scipy.sparse import diags
from scipy.interpolate import interp1d
from scipy.integrate import quad
from joblib import Parallel, delayed
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# =============================================
# 1. Parámetros Físicos Optimizados
# =============================================
D0 = 2e-9           # Coeficiente de difusión [m²/s]
R = 1e-6            # Radio del cilindro [m]
z_c = 10e-9         # Altura crítica [m]
lambda_param = 0.2  # Parámetro de decaimiento
delta = 1e-9        # Ancho de transición
n_modes_radial = 8  # Modos radiales
m_max = 10          # Modos axiales
N_z = 600           # Nodos en z
alpha_grid = 8.0     # Parámetro de malla

# =============================================
# 2. Función de Difusión Global
# =============================================
def D(z_val):
    """Perfil de difusión completo"""
    S = 0.5*(1 + np.tanh((z_val - z_c)/delta))
    return D0*(np.exp(-lambda_param*z_val)*(1 - S) + S)

# =============================================
# 3. Generación de Malla
# =============================================
j = np.linspace(0, 1, N_z)
z = z_c * (1 + np.sinh(alpha_grid*(j - 0.5)) / np.sinh(0.5*alpha_grid))
D_values = D(z)

# =============================================
# 4. Solución Radial
# =============================================
alpha_zeros = jn_zeros(0, n_modes_radial)
alpha_n = alpha_zeros/R
rho_plot = np.linspace(0.1*R, 0.9*R, 80)
J0_table = jv(0, alpha_n[:, None]*rho_plot)

# =============================================
# 5. Solución Axial con Manejo de Errores
# =============================================
def build_system(n):
    try:
        alpha = alpha_n[n]
        h = np.diff(z)
        N = len(z)
        
        main_diag = np.zeros(N)
        upper_diag = np.zeros(N-1)
        lower_diag = np.zeros(N-1)
        
        for j in range(1, N-1):
            h_prev = h[j-1]
            h_next = h[j]
            
            D_prev = 2*D_values[j]*D_values[j-1]/(D_values[j] + D_values[j-1] + 1e-30)
            D_next = 2*D_values[j]*D_values[j+1]/(D_values[j] + D_values[j+1] + 1e-30)
            
            h_avg = 0.5*(h_prev + h_next)
            
            main_diag[j] = D_prev/(h_prev*h_avg) + D_next/(h_next*h_avg) + alpha**2*D_values[j]
            upper_diag[j] = -D_next/(h_next*h_avg)
            lower_diag[j-1] = -D_prev/(h_prev*h_avg)
        
        main_diag[0] = D_values[0]/h[0] + 1e-10*D0
        upper_diag[0] = -D_values[0]/h[0]
        main_diag[-1] = 1.0
        
        A = diags([main_diag, upper_diag, lower_diag], [0, 1, -1])
        eigenvalues, eigenvectors = eigsh(A, k=m_max, which='SA', tol=1e-6, maxiter=15000)
        return (n, (eigenvalues, eigenvectors))
    
    except ArpackNoConvergence:
        print(f"Advertencia: Modo radial {n} no convergió. Usando ceros.")
        return (n, (np.zeros(m_max), np.zeros((N_z, m_max))))

# Cálculo paralelo seguro
results = Parallel(n_jobs=4, verbose=1)(delayed(build_system)(n) for n in range(n_modes_radial))

# Organizar resultados
eigenvalues_dict = {}
eigenvectors_dict = {}
for n, (evals, evecs) in sorted(results, key=lambda x: x[0]):
    eigenvalues_dict[n] = evals
    eigenvectors_dict[n] = evecs

# =============================================
# 6. Visualización de Modos (Corregida)
# =============================================
def safe_divide(a, b):
    return np.divide(a, b, out=np.zeros_like(a), where=b!=0)

# Modos radiales
plt.figure(figsize=(10, 6))
for n in range(4):
    mode = J0_table[n]
    plt.plot(rho_plot/R, safe_divide(mode, np.max(np.abs(mode))), 
            label=f'n={n+1}', linewidth=1.5)
plt.xlabel(r'$\rho/R$', fontsize=12)
plt.ylabel('Amplitud Normalizada', fontsize=12)
plt.title('Modos Radiales de Bessel', fontsize=14)
plt.legend()
plt.grid(alpha=0.4)
plt.savefig('modos_radiales.pdf', bbox_inches='tight')
plt.close()

# Modos axiales con división segura
fig, axs = plt.subplots(2, 2, figsize=(12, 8))
for idx, (n, m) in enumerate([(0,0), (0,1), (1,0), (1,1)]):
    ax = axs[idx//2, idx%2]
    mode = eigenvectors_dict[n][:,m]
    norm_mode = safe_divide(mode, np.max(np.abs(mode)))
    ax.plot(z*1e9, norm_mode, color='darkred')
    ax.set_xlabel('z (nm)', fontsize=10)
    ax.set_title(f'Modo (n={n}, m={m})', fontsize=12)
    ax.grid(alpha=0.4)
plt.tight_layout()
plt.savefig('modos_axiales.pdf', bbox_inches='tight')
plt.close()

# =============================================
# 7. Cálculo de Coeficientes A_nm
# =============================================
Z_interps = {}
for n in range(n_modes_radial):
    Z_interps[n] = [interp1d(z, eigenvectors_dict[n][:,m], 
                     bounds_error=False, fill_value=0.0) 
                   for m in range(m_max)]

def compute_A_nm(n):
    Z_norms = np.array([quad(lambda z_val: Z_interps[n][m](z_val)**2/D(z_val), 
                      z[0], z[-1], limit=300)[0] 
                   for m in range(m_max)])
    
    Z_m_z0 = np.array([Z_interps[n][m](15e-9) for m in range(m_max)])
    J0_rho0 = jv(0, alpha_n[n]*0.75e-6)
    
    valid = (Z_norms > 1e-25) & (np.abs(Z_m_z0) > 1e-12)
    return (J0_rho0 * Z_m_z0 * valid) / (2*np.pi*0.75e-6 * (Z_norms + 1e-30))

A_nm = Parallel(n_jobs=4)(delayed(compute_A_nm)(n) for n in range(n_modes_radial))

# =============================================
# 8. Densidad de Probabilidad 3D (Corregida)
# =============================================
def plot_density_3d(t, n_points=30):
    rho_vals = np.linspace(0.2 * R, 0.8 * R, n_points)
    z_vals = np.linspace(8e-9, 12e-9, n_points)
    Rg, Zg = np.meshgrid(rho_vals, z_vals)
    P = np.zeros_like(Rg)

    # Calcular el factor de decaimiento con clipping
    # decay = np.array([np.exp(np.clip(-((alpha_n[n]**2 * D0 + eigenvalues_dict[n]) * t), -700, 700)) for n in range(n_modes_radial)])  # Shape: (n_modes_radial, m_max)
    decay = np.array([np.exp(np.clip(-(eigenvalues_dict[n] * t),-700,700)) for n in range(n_modes_radial)])
                   
    # Crear arreglo 4D para las evaluaciones de los eigenvectores en z_vals
    Z = np.empty((len(z_vals), n_modes_radial, m_max))
    for j, z_val in enumerate(z_vals):
        for n in range(n_modes_radial):
            for m in range(m_max):
                Z[j, n, m] = Z_interps[n][m](z_val)

    A_nm_arr = np.array(A_nm)  # Asegurar el tipo

    # Calcular la densidad P en cada punto de la malla (Rho, Zg)
    for i, rho in enumerate(rho_vals):
        J0 = jv(0, alpha_n * rho)  # (n_modes_radial,)
        for j, z_val in enumerate(z_vals):
            f = (3 * z_val**2 - rho**2) / (rho**2 + z_val**2 + 1e-30)**2.5
            sum_term = np.sum(A_nm_arr * J0[:, None] * Z[j] * decay)
            # Verificar y forzar a 0 si el valor no es finito.
            if not np.isfinite(sum_term):
                sum_term = 0.0
            P[j, i] = np.abs(sum_term)
    
    # Limpiar la matriz P para asegurarse de no tener NaN o Inf
    P = np.nan_to_num(P, nan=0.0, posinf=1e40, neginf=0.0)
    
    # Calcular incrementos para cuadratura (para referencia en el escalado, aunque no afectan la visualización)
    dx = (0.8 * R - 0.2 * R) / n_points
    dz = (12e-9 - 8e-9) / n_points
    # Opcional: limitar los valores máximos de P para evitar escalas demasiado grandes
    P = np.clip(P, 0, 1e8)
    
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')
    # Usar el arreglo P corregido
    surf = ax.plot_surface(Rg * 1e6, Zg * 1e9, P, cmap='plasma',
                           rstride=1, cstride=1, alpha=0.9)
    ax.set_xlabel(r'$\rho$ (μm)', fontsize=12)
    ax.set_ylabel('z (nm)', fontsize=12)
    ax.set_zlabel('P(ρ,z,t)', fontsize=12)
    plt.title(f'Densidad en t = {t:.1e} s', fontsize=14)
    fig.colorbar(surf, shrink=0.6, aspect=20)
    plt.savefig(f'densidad_3d_t_{t:.0e}.pdf', bbox_inches='tight')
    plt.close()

for t in [1e-9, 1e-6, 1e-3]:
    plot_density_3d(t)

# =============================================
# 9. Función de Correlación G(t) Corregida y Optimizada
# =============================================
def compute_G(t, N_quad=12):
    # Definir nodos de cuadratura para ρ y z
    rho_nodes = np.linspace(0.3 * R, 0.7 * R, N_quad)
    z_nodes = np.linspace(9e-9, 11e-9, N_quad)
    total = 0.0

    # Calcular el factor de decaimiento para cada modo radial y axial con clipping
    # decay = np.array([np.exp(np.clip(-((alpha_n[n]**2 * D0 + eigenvalues_dict[n]) * t), -700, 700)) for n in range(n_modes_radial)])  # Shape: (n_modes_radial, m_max)
    decay = np.array([np.exp(np.clip(-(eigenvalues_dict[n] * t),-700,700)) for n in range(n_modes_radial)])
    # Convertir A_nm a arreglo NumPy
    A_nm_arr = np.array(A_nm)  # Shape: (n_modes_radial, m_max)

    # Precalcular las evaluaciones de los eigenvectores en los nodos de cuadratura en z
    Z_eval = np.empty((N_quad, n_modes_radial, m_max))
    for j, z_val in enumerate(z_nodes):
        for n in range(n_modes_radial):
            for m in range(m_max):
                Z_eval[j, n, m] = Z_interps[n][m](z_val)

    # Bucle doble para la cuadratura en ρ y z
    for rho in rho_nodes:
        # Calcular J0 para cada modo radial
        J0 = jv(0, alpha_n * rho)  # Shape: (n_modes_radial,)
        for j, z_val in enumerate(z_nodes):
            # Factor de forma f(ρ,z)
            f = (3 * z_val**2 - rho**2) / (rho**2 + z_val**2 + 1e-30)**2.5
            # Sumar contribuciones: 
            #   J0[:, None] -> (n_modes_radial, 1)
            #   Z_eval[j] -> (n_modes_radial, m_max)
            #   decay -> (n_modes_radial, m_max)
            #   A_nm_arr -> (n_modes_radial, m_max)
            sum_term = np.sum(A_nm_arr * J0[:, None] * Z_eval[j] * decay)
            # Evitar NaN: si sum_term es inválido, forzarlo a 0.
            if not np.isfinite(sum_term):
                sum_term = 0.0
            total += f * sum_term

    # Limpieza del total y uso de valores razonables
    total = np.nan_to_num(total, nan=0.0, posinf=1e40, neginf=0.0)
    # Calcular incrementos (dx, dz) en función de los intervalos de cuadratura
    dx = (0.7 * R - 0.3 * R) / N_quad
    dz = (11e-9 - 9e-9) / N_quad
    result = (5 / (16 * np.pi)) * (total * dx * dz)
    # Asegurarse de que el resultado esté dentro de un rango razonable
    return max(result, 1e-40)

# ## DEBUG ##
# import sys
# def compute_G(t, N_quad=12):
#     with open("debug_log.txt", "a") as f:
#         f.write(f"Calculando G(t) para t = {t}\n")
#     rho_nodes = np.linspace(0.3 * R, 0.7 * R, N_quad)
#     z_nodes = np.linspace(9e-9, 11e-9, N_quad)
#     total = 0.0
#     decay = np.array([
#         np.exp(np.clip(-((alpha_n[n]**2 * D0 + eigenvalues_dict[n]) * t), -700, 700))
#         for n in range(n_modes_radial)
#     ])
#     with open("debug_log.txt", "a") as f:
#         f.write(f"Decay: {decay}\n")
#     A_nm_arr = np.array(A_nm)
#     Z_eval = np.empty((N_quad, n_modes_radial, m_max))
#     for j, z_val in enumerate(z_nodes):
#         for n in range(n_modes_radial):
#             for m in range(m_max):
#                 Z_eval[j, n, m] = Z_interps[n][m](z_val)
#     for rho in rho_nodes:
#         J0 = jv(0, alpha_n * rho)
#         for j, z_val in enumerate(z_nodes):
#             f = (3 * z_val**2 - rho**2) / (rho**2 + z_val**2 + 1e-30)**2.5
#             sum_term = np.sum(A_nm_arr * J0[:, None] * Z_eval[j] * decay)
#             if not np.isfinite(sum_term):
#                 sum_term = 0.0
#             total += f * sum_term
#     dx = (0.7 * R - 0.3 * R) / N_quad
#     dz = (11e-9 - 9e-9) / N_quad
#     result = (5 / (16 * np.pi)) * (total * dx * dz)
#     with open("debug_log.txt", "a") as f:
#         f.write(f"Total*dx*dz: {total*dx*dz} => G(t): {result}\n")
#     return max(result, 1e-40)

# Prueba la función de depuración
# debug_value = compute_G(1e-9)
# print("Valor de G(t) en modo debug:", debug_value)


# Ejecución del cálculo de G(t) para un conjunto de tiempos
t_values = np.logspace(-9, -2, 25)
G_values = Parallel(n_jobs=4)(delayed(compute_G)(t) for t in t_values)

np.savetxt('G_vs_t.dat', np.c_[t_values, G_values], 
               header='Tiempo(s) G(t)', fmt='%.3e')

plt.figure(figsize=(10, 6))
plt.loglog(t_values, G_values, 'o-', color='navy', markersize=6, 
           linewidth=1.5, label='Simulación')
plt.plot(t_values, 1e-8 * t_values**-1.5, '--', color='crimson', 
         linewidth=1.5, label=r'Teoría $t^{-3/2}$')
plt.xlabel('Tiempo (s)', fontsize=12)
plt.ylabel('G(t)', fontsize=12)
plt.title('Función de Correlación Nano-NMR', fontsize=14)
plt.legend()
plt.grid(which='both', alpha=0.4)
plt.savefig('correlacion_nanoNMR.pdf', bbox_inches='tight', dpi=300)
plt.close()

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   8 out of   8 | elapsed:   10.1s finished
